# Business Queries – SQL to Notebook

All five queries are copied verbatim from `business_queries.sql` and run against a SQLite database.

**Rules shared by every query** (from the original file):
- Valid transaction records only: invalid `order_date`, zero/negative `quantity` and non-positive `unit_price` are excluded.
- Net revenue = `quantity * unit_price * (1 - discount)`, excluding returned orders (`returned = 0`).

⚠️ Set `DB_PATH` below. The tables `orders`, `customers` and `products` must exist in that database.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = "your_database.db"   # <-- change this

pd.options.display.float_format = "{:,.2f}".format
conn = sqlite3.connect(DB_PATH)

def run(sql):
    return pd.read_sql_query(sql, conn)

## 1. Total net revenue

In [ ]:
query_1 = """
WITH clean_orders AS (
    SELECT *, quantity * unit_price * (1 - discount) AS line_revenue
    FROM orders
    WHERE date(order_date) IS NOT NULL
      AND quantity > 0
      AND unit_price > 0
)
SELECT ROUND(SUM(line_revenue), 2) AS total_net_revenue
FROM clean_orders
WHERE returned = 0
"""

df_1 = run(query_1)
df_1

## 2. Top 10 customers by total spending

In [ ]:
query_2 = """
WITH clean_orders AS (
    SELECT *, quantity * unit_price * (1 - discount) AS line_revenue
    FROM orders
    WHERE date(order_date) IS NOT NULL
      AND quantity > 0
      AND unit_price > 0
), clean_customers AS (
    SELECT *, CASE UPPER(TRIM(city))
        WHEN 'KARACHI' THEN 'Karachi'
        WHEN 'LAHORE' THEN 'Lahore'
        WHEN 'ISLAMABAD' THEN 'Islamabad'
        WHEN 'RAWALPINDI' THEN 'Rawalpindi'
        WHEN 'FAISALABAD' THEN 'Faisalabad'
        WHEN 'MULTAN' THEN 'Multan'
        WHEN 'PESHAWAR' THEN 'Peshawar'
        WHEN 'HYDERABAD' THEN 'Hyderabad'
        WHEN 'GUJRANWALA' THEN 'Gujranwala'
        WHEN 'QUETTA' THEN 'Quetta'
        WHEN 'SIALKOT' THEN 'Sialkot'
        WHEN 'BAHAWALPUR' THEN 'Bahawalpur'
        WHEN 'SARGODHA' THEN 'Sargodha'
        ELSE TRIM(city)
    END AS clean_city FROM customers
)
SELECT c.customer_name, c.clean_city AS city,
       COUNT(DISTINCT o.order_id) AS number_of_orders,
       ROUND(SUM(o.line_revenue), 2) AS total_spending
FROM clean_orders o
JOIN clean_customers c ON c.customer_id = o.customer_id
WHERE o.returned = 0
GROUP BY c.customer_id, c.customer_name, c.clean_city
ORDER BY total_spending DESC
LIMIT 10
"""

df_2 = run(query_2)
df_2

## 3. Category-wise revenue, order count and quantity sold

In [ ]:
query_3 = """
WITH clean_orders AS (
    SELECT *, quantity * unit_price * (1 - discount) AS line_revenue
    FROM orders
    WHERE date(order_date) IS NOT NULL
      AND quantity > 0
      AND unit_price > 0
), clean_products AS (
    SELECT *, CASE UPPER(TRIM(category))
        WHEN 'FASHION' THEN 'Fashion'
        WHEN 'ELECTRONICS' THEN 'Electronics'
        WHEN 'HOME & KITCHEN' THEN 'Home & Kitchen'
        WHEN 'BEAUTY' THEN 'Beauty'
        WHEN 'GROCERIES' THEN 'Groceries'
        WHEN 'SPORTS & FITNESS' THEN 'Sports & Fitness'
        WHEN 'BOOKS & STATIONERY' THEN 'Books & Stationery'
        WHEN 'BABY & KIDS' THEN 'Baby & Kids'
        ELSE TRIM(category)
    END AS clean_category FROM products
)
SELECT p.clean_category AS category,
       ROUND(SUM(o.line_revenue), 2) AS net_revenue,
       COUNT(DISTINCT o.order_id) AS order_count,
       SUM(o.quantity) AS quantity_sold
FROM clean_orders o
JOIN clean_products p ON p.product_id = o.product_id
WHERE o.returned = 0
GROUP BY p.clean_category
ORDER BY net_revenue DESC
"""

df_3 = run(query_3)
df_3

In [ ]:
ax = df_3.plot.bar(x="category", y="net_revenue", legend=False, figsize=(9, 4), title="Net revenue by category")
ax.set_ylabel("Net revenue"); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 4. Monthly net revenue trend

In [ ]:
query_4 = """
WITH clean_orders AS (
    SELECT *, quantity * unit_price * (1 - discount) AS line_revenue
    FROM orders
    WHERE date(order_date) IS NOT NULL
      AND quantity > 0
      AND unit_price > 0
)
SELECT strftime('%Y-%m', order_date) AS month,
       ROUND(SUM(line_revenue), 2) AS net_revenue
FROM clean_orders
WHERE returned = 0
GROUP BY month
ORDER BY month
"""

df_4 = run(query_4)
df_4

In [ ]:
ax = df_4.plot(x="month", y="net_revenue", marker="o", legend=False, figsize=(10, 4), title="Monthly net revenue")
ax.set_ylabel("Net revenue"); plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 5. Top 5 products by net revenue

In [ ]:
query_5 = """
WITH clean_orders AS (
    SELECT *, quantity * unit_price * (1 - discount) AS line_revenue
    FROM orders
    WHERE date(order_date) IS NOT NULL
      AND quantity > 0
      AND unit_price > 0
)
SELECT p.product_name, p.category,
       ROUND(SUM(o.line_revenue), 2) AS net_revenue,
       SUM(o.quantity) AS quantity_sold,
       COUNT(DISTINCT o.order_id) AS order_count
FROM clean_orders o
JOIN products p ON p.product_id = o.product_id
WHERE o.returned = 0
GROUP BY p.product_id, p.product_name, p.category
ORDER BY net_revenue DESC
LIMIT 5
"""

df_5 = run(query_5)
df_5

## Sanity checks (not in the original SQL)

The five queries assume clean keys, a fractional `discount`, and no duplicate rows. Nothing in the SQL verifies that. These cells test those assumptions; if any fails, the numbers above are wrong in a way that no error message will reveal.

### Reconciliation
Q1, Q3 and Q4 should all total the same figure. Q2 and Q5 are top-N so they can't be compared directly. A gap in Q3 means orders with an orphan `product_id` are being dropped by the inner join.

In [ ]:
checks = pd.DataFrame({
    "source": ["Q1 total", "Q3 sum of categories", "Q4 sum of months"],
    "net_revenue": [df_1.iloc[0, 0], df_3["net_revenue"].sum(), df_4["net_revenue"].sum()],
})
checks["diff_vs_Q1"] = checks["net_revenue"] - checks.loc[0, "net_revenue"]
checks

### Orphan keys and duplicate IDs

In [ ]:
diag = run("""
SELECT 'orders with no matching product'  AS check_name, COUNT(*) AS n FROM orders o
  WHERE NOT EXISTS (SELECT 1 FROM products p WHERE p.product_id = o.product_id)
UNION ALL
SELECT 'orders with no matching customer', COUNT(*) FROM orders o
  WHERE NOT EXISTS (SELECT 1 FROM customers c WHERE c.customer_id = o.customer_id)
UNION ALL
SELECT 'duplicate product_id values in products', COUNT(*) FROM
  (SELECT product_id FROM products GROUP BY product_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'duplicate customer_id values in customers', COUNT(*) FROM
  (SELECT customer_id FROM customers GROUP BY customer_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'order_id values appearing on >1 row', COUNT(*) FROM
  (SELECT order_id FROM orders GROUP BY order_id HAVING COUNT(*) > 1)
""")
diag

*Duplicate `product_id`/`customer_id` rows fan out the joins and inflate revenue. Repeated `order_id` is only a problem if the rows are true duplicates rather than multi-line orders.*

### Discount, returned flag and date coverage

In [ ]:
run("""
SELECT
  SUM(discount IS NULL)                         AS null_discount,
  SUM(discount < 0 OR discount > 1)             AS discount_outside_0_1,
  MIN(discount) AS min_discount, MAX(discount) AS max_discount,
  SUM(returned IS NULL)                         AS null_returned,
  SUM(date(order_date) IS NULL)                 AS rows_dropped_bad_date,
  SUM(quantity <= 0 OR unit_price <= 0)         AS rows_dropped_qty_or_price,
  COUNT(*)                                      AS total_rows
FROM orders
""")

*A NULL `discount` makes `line_revenue` NULL, so the row silently vanishes from every SUM. A NULL `returned` fails `returned = 0` and is also dropped. A discount stored as a percentage (e.g. 10 instead of 0.10) produces negative revenue.*

### Raw category values (Q5 does not clean them)
Q3 normalises `category` but Q5 outputs the raw column. If this shows case/whitespace variants, Q5's `category` column is inconsistent with Q3.

In [ ]:
run("""
SELECT category AS raw_category, TRIM(category) <> category AS has_whitespace, COUNT(*) AS products
FROM products GROUP BY category ORDER BY UPPER(TRIM(category))
""")

In [ ]:
conn.close()